# 面向对象高级编程

## ___\_\_slots\_\____
> 类实例化后，可以给实例化后的变量绑定任何属性和方法
>
> 但是被绑定的属性和方法只在该实例上起作用
>
> 如果想要所有实例绑定同一个方法，可以给类绑定方法
>
> 定义一个特殊的__slots__变量，来限制该class实例能添加的属性
>
> \_\_slots\_\_ = ('name', 'age') # 用tuple定义允许绑定的属性名称
>
> \_\_slots\_\_定义的属性仅对当前类实例起作用，对继承的子类是不起作用的
>


In [7]:
from IPython.core.guarded_eval import dict_items


class Student(object):
    __slots__ = ("name", "age")
    pass
s = Student()
s.name = "Mic"
def say_hello(stu):
    print(f"{stu.name} says Hello")
# s.greet = say_hello(s)
# if(hasattr(s, "greet")):
#     s.greet
print(s.name)

del s.name
print(hasattr(s, "name"))

Student.age = 20
print(s.age)

Mic
False
20


## ___@property___

> 用于简化set,get方法
>
> ***@property*** ——1）把getter方法变成属性；2）会创建另一个装饰器 ***@xxx.setter***
>
> ***@xxx.setter*** 负责把setter方法变成属性赋值
>
> 1. 只定义getter方法，不定义setter方法就是一个只读属性
>
> 2. 属性的方法名不能和实例变量重名，比如属性方法为score,实例就要写成_score，否则会导致无限递归
>

In [11]:
class Student(object):
    @property
    def score(self):
        # 不能写成self.score
        # return self.score
        return self._score

    @score.setter
    def score(self, value):
        if not isinstance(value, int):
            raise ValueError('score must be an integer!')
        if value < 0 or value > 100:
            raise ValueError('score must between 0 ~ 100!')
        self._score = value
s=Student()
s.score=100
print(s.score)
# s.score=101

100


## ___多重继承___
- **MixIn** ——继承多个父类

In [14]:
class Animal(object):
    pass

# 大类:
class Mammal(Animal):
    pass

class Bird(Animal):
    pass

# 各种动物:
class Dog(Mammal):
    pass

class Bat(Mammal):
    pass

class Parrot(Bird):
    pass

class Ostrich(Bird):
    pass
class Runnable(object):
    def run(self):
        print('Running...')

class Flyable(object):
    def fly(self):
        print('Flying...')
class Dog(Mammal, Runnable):
    pass
d = Dog()
d.run()

Running...


## ***定制类***

1. **\_\_str\_\_**      print打印结果

In [18]:
class Student(object):
    def __init__(self, name):
       self.name = name
print(Student("mic"))
class Student(object):
    def __init__(self, name):
       self.name = name
    def __str__(self):
        return 'Student object (name: %s)' % self.name
print(Student("mic"))
Student("mic")

Student object (name: mic)


2. **\_\_repr\_\_**     开发者看到的，用于调试

In [19]:
class Student(object):
    def __init__(self, name):
       self.name = name
Student("mic")

In [20]:
class Student(object):
    def __init__(self, name):
       self.name = name
    def __repr__(self):
        return 'Student object (name: %s)' % self.name
Student("mic")

Student object (name: mic)

3. **\_\_iter\_\_, \_\_getitem\_\_, \_\_setitem\_\_, \_\_delitem\_\_,**
> 该方法返回一个迭代对象，然后，Python的for循环就会不断调用该迭代对象的__next__()方法拿到循环的下一个值，
>
> 直到遇到StopIteration错误时退出循环
>
> **\_\_getitem\_\_, \_\_setitem\_\_, \_\_delitem\_\_**
>

In [47]:
class Fib(object):
    def __init__(self):
        self.a, self.b = 0, 1 # 初始化两个计数器a，b

    def __iter__(self):
        return self # 实例本身就是迭代对象，故返回自己

    def __next__(self):
        self.a, self.b = self.b, self.a + self.b # 计算下一个值
        if self.a > 100: # 退出循环的条件
            raise StopIteration()
        return self.a # 返回下一个值
    def __getitem__(self, n):
        if isinstance(n, int): # n是索引
            a, b = 1, 1
            for x in range(n):
                a, b = b, a + b
            return a
        if isinstance(n, slice): # n是切片
            start = n.start
            stop = n.stop
            if start is None:
                start = 0
            a, b = 1, 1
            L = []
            for x in range(stop):
                if x >= start:
                    L.append(a)
                a, b = b, a + b
            return L
my_fib = Fib()
for n in my_fib:
    print(n)
print('-'*75)
print(Fib()[:5])

1
1
2
3
5
8
13
21
34
55
89
---------------------------------------------------------------------------
[1, 1, 2, 3, 5]


4. **\_\_getattr\_\_**
> 只有在没有找到属性的情况下，才调用__getattr__
>
> __getattr__可以把一个类的所有属性和方法调用全部动态化处理了
>

In [26]:
class Student(object):
    def __init__(self):
        self.name = 'Michael'

    def __getattr__(self, attr):
        if attr=='score':
            return 99# 返回具体值
        if attr=='age':
            return lambda: 25# 返回函数
        if attr=='name':
            return "I`m no one"# 这个属性是有的，所以不会执行这个函数
stu = Student()
print(stu.score)
print(stu.age())
print(stu.name)
# 这一步删除掉了name这个属性，所以会执行__getattr__
del stu.name
print(stu.name)

99
25
Michael
I`m no one


In [34]:
class Chain(object):
    def __init__(self, path=''):
        self._path = path

    def __getattr__(self, path):
        return Chain(f'{self._path}//{path}')

    def __str__(self):
        return self._path
    __repr__ = __str__
print(Chain().status)
print(Chain().status.user)
print(Chain().status.user.timeline.list)

//status
//status//user
//status//user//timeline//list
---------------------------------------------------------------------------
GET /users/repos


5. **\_\_call\_\_**
> 任何类，只需要定义一个__call__()方法，就可以直接对实例进行调用
>
> 把实例对象看成函数，把函数看成对象实例
>

In [41]:
class Student(object):
    def __init__(self, name="anonymous"):
        self.name = name

    def __call__(self):
        print('My name is %s.' % self.name)
s=Student("mic")
s()
print(callable(Student()),callable([1,2]))

My name is mic.
True False


In [44]:
class Github(object):
    def __init__(self, path=''):
        self._path = path

    def __getattr__(self, path):
        return Github(f'{self._path}/{path}')
    def __call__(self, args):
        return Github(f'{self._path}/{args}')
    def GetRequest(self):
        return f"GET {self._path}"
my_git = Github().users("Gallnut").repos
print(my_git.GetRequest())

GET /users/Gallnut/repos


## ***枚举类***

In [65]:
from enum import Enum

Month = Enum('Month', ('Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'))
for name, member in Month.__members__.items():
    print(name, '=>', member, ',', member.value)
print(type(Month.__members__))
print(type(Month.__members__.items()))
month_dict = Month.__members__.items()
print(isinstance(month_dict, dict_items))

Jan => Month.Jan , 1
Feb => Month.Feb , 2
Mar => Month.Mar , 3
Apr => Month.Apr , 4
May => Month.May , 5
Jun => Month.Jun , 6
Jul => Month.Jul , 7
Aug => Month.Aug , 8
Sep => Month.Sep , 9
Oct => Month.Oct , 10
Nov => Month.Nov , 11
Dec => Month.Dec , 12
<class 'mappingproxy'>
<class 'dict_items'>


NameError: name 'dict_items' is not defined

In [51]:
from enum import Enum, unique
# @unique可以保证没有重复值
@unique
class Weekday(Enum):
    Sun = 0 # Sun的value被设定为0
    Mon = 1
    Tue = 2
    Wed = 3
    Thu = 4
    Fri = 5
    Sat = 6
day1 = Weekday.Mon
print(day1.value)

1
